# 11.2 检索 (Retrieval)

检索是RAG的核心环节，从知识库中找到与查询最相关的文档。检索质量直接决定生成质量。

本节涵盖：
- BM25稀疏检索
- 稠密检索（双编码器）
- 混合检索
- 重排序（Cross-Encoder）

## 1. BM25稀疏检索

**目的**：基于词项匹配的经典检索算法，是稀疏检索的工业标准

**BM25公式**：
```
score(D, Q) = Σ_{qi∈Q} IDF(qi) × f(qi,D) × (k1+1) / (f(qi,D) + k1×(1-b+b×|D|/avgdl))
```

**关键参数**：
- **f(qi,D)**：词qi在文档D中的词频（TF）
- **IDF(qi)**：逆文档频率，log((N-n(qi)+0.5)/(n(qi)+0.5))
- **k1**：词频饱和参数，通常1.2-2.0，控制TF的边际递减
- **b**：文档长度归一化，通常0.75，b=0忽略长度，b=1完全归一化
- **avgdl**：平均文档长度

**优势**：精确匹配、可解释性强、无需训练、计算快
**劣势**：无法理解语义（"汽车"≠"轿车"）

In [ ]:
import math
from collections import Counter

class BM25:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.documents = documents
        self.doc_lens = [len(doc.split()) for doc in documents]
        self.avgdl = sum(self.doc_lens) / len(documents)
        self.n_docs = len(documents)
        self.doc_freqs = Counter()
        self.doc_tfs = []

        for doc in documents:
            tf = Counter(doc.lower().split())
            self.doc_tfs.append(tf)
            for term in tf:
                self.doc_freqs[term] += 1

    def _idf(self, term):
        n = self.doc_freqs.get(term, 0)
        return math.log((self.n_docs - n + 0.5) / (n + 0.5) + 1)

    def score(self, query, doc_idx):
        query_terms = query.lower().split()
        doc_tf = self.doc_tfs[doc_idx]
        doc_len = self.doc_lens[doc_idx]
        score = 0.0
        for term in query_terms:
            tf = doc_tf.get(term, 0)
            idf = self._idf(term)
            numerator = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
            score += idf * numerator / denominator
        return score

    def search(self, query, top_k=5):
        scores = [(i, self.score(query, i)) for i in range(self.n_docs)]
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

documents = [
    "Machine learning is a subset of artificial intelligence",
    "Deep learning uses neural networks with multiple layers",
    "Natural language processing focuses on human language",
    "Machine learning algorithms learn from training data",
    "Neural networks are inspired by biological neural networks",
    "Language models process and generate natural language text",
    "Artificial intelligence includes machine learning and deep learning",
    "Training data is essential for supervised machine learning",
]

bm25 = BM25(documents)

print('=== BM25 Sparse Retrieval ===')
queries = [
    "machine learning algorithms",
    "neural networks deep learning",
    "natural language processing",
]

for query in queries:
    results = bm25.search(query, top_k=3)
    print(f'\nQuery: "{query}"')
    for rank, (doc_idx, score) in enumerate(results):
        print(f'  Rank {rank+1}: score={score:.4f} - "{documents[doc_idx][:60]}"')

print(f'\nKey: BM25 uses TF-IDF with length normalization.')
print(f'IDF penalizes common words; k1 controls TF saturation; b normalizes document length.')
print(f'Production: use Elasticsearch/OpenSearch for distributed BM25 at scale.')

## 2. 稠密检索（双编码器）

**目的**：基于语义相似度检索，超越关键词匹配

**双编码器架构**：
- 查询编码器：将查询映射到向量 q = Encoder(query)
- 文档编码器：将文档映射到向量 d = Encoder(doc)
- 相似度：score = cosine_similarity(q, d)

**训练方式**：
- **对比学习**：正样本对（相关query-doc）拉近，负样本对推远
- **In-batch negatives**：同一batch内其他文档作为负样本
- **Hard negatives**：BM25检索的高分但不相关文档作为难负样本

**工业模型**：DPR、E5、BGE、GTE

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class DualEncoder(nn.Module):
    def __init__(self, vocab_size=3000, d_model=64, d_embed=32):
        super().__init__()
        self.q_embed = nn.Embedding(vocab_size, d_model)
        self.d_embed = nn.Embedding(vocab_size, d_model)
        self.q_encoder = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, d_embed))
        self.d_encoder = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, d_embed))

    def encode_query(self, q_ids):
        h = self.q_embed(q_ids).mean(dim=1)
        return F.normalize(self.q_encoder(h), dim=-1)

    def encode_doc(self, d_ids):
        h = self.d_embed(d_ids).mean(dim=1)
        return F.normalize(self.d_encoder(h), dim=-1)

model = DualEncoder()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

n_docs = 50
seq_len = 8
query_ids = torch.randint(0, 3000, (16, seq_len))
pos_doc_ids = torch.randint(0, 3000, (16, seq_len))
all_doc_ids = torch.randint(0, 3000, (n_docs, seq_len))

print('=== Dense Retrieval (Dual Encoder) Training ===')
for epoch in range(10):
    q_emb = model.encode_query(query_ids)
    pos_emb = model.encode_doc(pos_doc_ids)
    neg_idx = torch.randperm(n_docs)[:16]
    neg_emb = model.encode_doc(all_doc_ids[neg_idx])

    pos_scores = (q_emb * pos_emb).sum(dim=-1)
    neg_scores = (q_emb * neg_emb).sum(dim=-1)
    loss = F.relu(0.1 - pos_scores + neg_scores).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 3 == 0:
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}, pos_avg={pos_scores.mean():.4f}, neg_avg={neg_scores.mean():.4f}')

model.eval()
with torch.no_grad():
    q_emb = model.encode_query(query_ids[:3])
    d_emb = model.encode_doc(all_doc_ids)
    scores = q_emb @ d_emb.T
    for i in range(3):
        top3 = scores[i].topk(3)
        print(f'\nQuery {i} top-3: indices={top3.indices.tolist()}, scores={top3.values.tolist()}')

print(f'\nKey: Dual encoder enables fast retrieval via vector similarity.')
print(f'Contrastive learning with hard negatives improves retrieval quality.')
print(f'Production: pre-compute doc embeddings, only encode query at search time.')

## 3. 混合检索与重排序

**目的**：结合稀疏检索的精确匹配和稠密检索的语义理解

**混合检索**：
```
hybrid_score = α × dense_score + (1-α) × sparse_score
```
- α通常0.5-0.8，稠密检索权重更高
- 需要归一化两种分数到相同尺度（min-max或z-score）

**重排序（Reranking）**：
- 第一阶段：Bi-Encoder快速召回100-1000候选
- 第二阶段：Cross-Encoder精细排序top-10-20
- Cross-Encoder同时编码query和doc，精度更高但速度慢

**工业实践**：
- Elasticsearch做BM25 + 向量检索
- Cohere Rerank / BGE-Reranker做重排序
- 典型延迟：检索<50ms，重排序<100ms

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from collections import Counter

torch.manual_seed(42)

class CrossEncoder(nn.Module):
    def __init__(self, vocab_size=3000, d_model=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.encoder = nn.Sequential(
            nn.Linear(d_model * 2, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, query_ids, doc_ids):
        q_h = self.embed(query_ids).mean(dim=1)
        d_h = self.embed(doc_ids).mean(dim=1)
        combined = torch.cat([q_h, d_h], dim=-1)
        return self.encoder(combined).squeeze(-1)

documents = [
    "machine learning is a subset of artificial intelligence",
    "deep learning uses neural networks with multiple layers",
    "natural language processing focuses on human language",
    "machine learning algorithms learn from training data",
    "neural networks are inspired by biological neural networks",
    "language models process and generate natural language text",
    "artificial intelligence includes machine learning and deep learning",
    "training data is essential for supervised machine learning",
]

bm25 = BM25(documents)
dual_enc = DualEncoder()
cross_enc = CrossEncoder()

query = "machine learning training"
vocab = {w: i for i, w in enumerate(set(' '.join(documents).split()))}

def text_to_ids(text, vocab, max_len=10):
    tokens = text.lower().split()[:max_len]
    return torch.tensor([[vocab.get(t, 0) for t in tokens]])

print('=== Hybrid Retrieval + Reranking ===')
print(f'Query: "{query}"')

bm25_results = bm25.search(query, top_k=5)
print(f'\n--- Stage 1: BM25 Sparse Retrieval ---')
for rank, (idx, score) in enumerate(bm25_results):
    print(f'  Rank {rank+1}: score={score:.4f} - "{documents[idx][:50]}"')

sparse_scores = torch.tensor([s for _, s in bm25_results])
sparse_norm = (sparse_scores - sparse_scores.min()) / (sparse_scores.max() - sparse_scores.min() + 1e-8)

query_t = text_to_ids(query, vocab)
doc_indices = [idx for idx, _ in bm25_results]
doc_texts = [documents[idx] for idx in doc_indices]
doc_t = torch.cat([text_to_ids(d, vocab) for d in doc_texts])

with torch.no_grad():
    q_emb = dual_enc.encode_query(query_t.expand(len(doc_indices), -1))
    d_emb = dual_enc.encode_doc(doc_t)
    dense_scores = (q_emb * d_emb).sum(dim=-1)
    dense_norm = (dense_scores - dense_scores.min()) / (dense_scores.max() - dense_scores.min() + 1e-8)

alpha = 0.6
hybrid_scores = alpha * dense_norm + (1 - alpha) * sparse_norm

print(f'\n--- Stage 2: Hybrid Score (α={alpha}) ---')
for i, idx in enumerate(doc_indices):
    print(f'  Doc {idx}: sparse={sparse_norm[i]:.3f}, dense={dense_norm[i]:.3f}, hybrid={hybrid_scores[i]:.3f}')

with torch.no_grad():
    cross_scores = cross_enc(query_t.expand(len(doc_indices), -1), doc_t)

reranked_order = cross_scores.argsort(descending=True)
print(f'\n--- Stage 3: Cross-Encoder Reranking ---')
for rank, order_idx in enumerate(reranked_order.tolist()):
    doc_idx = doc_indices[order_idx]
    print(f'  Rank {rank+1}: cross_score={cross_scores[order_idx]:.4f} - "{documents[doc_idx][:50]}"')

print(f'\nKey: 3-stage pipeline: BM25 recall → Hybrid score → Cross-Encoder rerank.')
print(f'BM25 handles exact matches; dense retrieval handles semantic similarity.')
print(f'Cross-Encoder provides the most accurate ranking but is too slow for first-stage retrieval.')